# Program Input and Error Handling

*Accept text from outside a program, convert it deliberately, and recover from expected invalid values.*

Small programs cannot assume that every input has the expected type or range. Python's `input()` function always returns text, and `try`/`except` separates successful conversion from expected failure handling.

## `input()` Returns a String

The prompt is displayed to the user, and the entered characters are returned as a string. Defining the function below does not open a prompt until the function is called.

In [1]:
def ask_participant_count():
    raw_count = input("Number of participants: ")
    return int(raw_count)

print("Interactive function defined:", ask_participant_count.__name__)

Interactive function defined: ask_participant_count


In an interactive script, calling `ask_participant_count()` pauses for user input and converts the returned string with `int()`. The remaining examples use prepared strings so the notebook can run without pausing.

## `try` and `except`

Code that may fail belongs in `try`. A matching `except` handles the expected exception and lets the program continue.

In [2]:
def parse_participant_count(raw_count):
    try:
        return int(raw_count), None
    except ValueError as error:
        return None, str(error)

valid_count, valid_error = parse_participant_count("24")
invalid_count, invalid_error = parse_participant_count("twenty")

print("Valid result:", valid_count, valid_error)
print("Invalid result:", invalid_count, invalid_error)

Valid result: 24 None
Invalid result: None invalid literal for int() with base 10: 'twenty'


The valid text becomes an integer. Invalid numeric text raises `ValueError`, which the function catches and reports as data instead of stopping the program.

## Raising a Clear Error

`raise` reports that a value violates a rule even when its type is correct. The caller may catch that exception and decide how to respond.

In [3]:
def require_nonnegative(value):
    if value < 0:
        raise ValueError("value must be zero or greater")
    return value

try:
    accepted_value = require_nonnegative(-3)
except ValueError as error:
    accepted_value = None
    print("Validation error:", error)

print("Accepted value:", accepted_value)

Validation error: value must be zero or greater
Accepted value: None


The function raises a specific error at the invalid boundary. The surrounding handler converts the failure into a readable message and a missing result.

## `else` and `finally`

The `else` block runs only when `try` succeeds. The `finally` block runs whether the operation succeeds or fails, which is useful for cleanup or final status logging.

In [4]:
events = []

try:
    temperature_c = float("18.6")
except ValueError:
    parse_status = "invalid"
else:
    parse_status = "valid"
    events.append("temperature converted")
finally:
    events.append("parsing finished")

print("Parse status:", parse_status)
print("Events:", events)

Parse status: valid
Events: ['temperature converted', 'parsing finished']


The conversion succeeds, so `else` records a valid result. `finally` adds the completion event regardless of that result.

## Use Cases

Input and exception handling commonly support form values, menu selections, imported text fields, and domain validation.

### Numeric Form Values

Form fields arrive as text and must be converted before arithmetic.

In [5]:
raw_fee = "12500"

try:
    fee = int(raw_fee)
    fee_message = f"Fee accepted: {fee} won"
except ValueError:
    fee = None
    fee_message = "Fee must be a whole number"

print(fee_message)

Fee accepted: 12500 won


The prepared form value is valid integer text, so the program stores a numeric fee that later calculations can use.

### Menu Selections

Membership validation checks whether a text choice belongs to the allowed set.

In [6]:
raw_choice = "JSON"
allowed_choices = {"csv", "json"}
normalized_choice = raw_choice.strip().lower()

if normalized_choice in allowed_choices:
    choice_status = "accepted"
else:
    choice_status = "unsupported"

print("Choice:", normalized_choice)
print("Status:", choice_status)

Choice: json
Status: accepted


Normalization handles capitalization and outside spaces before the set membership check is applied.

### Imported Numeric Text

Several text values can be converted independently so one bad field does not hide which item failed.

In [7]:
raw_measurements = ["39.1", "missing", "40.3"]
converted_measurements = []
conversion_errors = []

for position, raw_value in enumerate(raw_measurements):
    try:
        converted_measurements.append(float(raw_value))
    except ValueError:
        conversion_errors.append((position, raw_value))

print("Converted:", converted_measurements)
print("Errors:", conversion_errors)

Converted: [39.1, 40.3]
Errors: [(1, 'missing')]


The two numeric fields are preserved, and the invalid field is recorded with its position for later review.

### Domain Constraints

Parsing answers “is this text a number?” Domain validation answers “is this number allowed here?”

In [8]:
def validate_attendance_rate(rate):
    if not 0 <= rate <= 1:
        raise ValueError("attendance rate must be between 0 and 1")
    return rate

for candidate in [0.9, 1.2]:
    try:
        print("Accepted rate:", validate_attendance_rate(candidate))
    except ValueError as error:
        print("Rejected rate:", candidate, "-", error)

Accepted rate: 0.9
Rejected rate: 1.2 - attendance rate must be between 0 and 1


The valid proportion is accepted and the out-of-range value receives a specific explanation. Type conversion and domain validation solve different problems.

Use `try` only around the operation expected to fail, catch specific exception types, and keep the successful result separate from the error message. Do not use a broad empty `except` block because it can hide unrelated programming mistakes.